# House Prices: repeated-CV feature-combination submissions

## tl;dr

상위 두 feature 조합의 submission CSV 두 개를 생성하고 형식을 검증했다.
각 파일은 해당 후보의 restored checkpoint 15개만 mean-log 평균하며 서로
blend하지 않는다. 두 파일 모두 1,459행, Id 순서 일치, 결측·비유한·음수
예측 0건이다. Kaggle public/private 점수는 아직 unverified다.


## Context & Methods

### Key Assumptions

- 후보 1은 TotalSF + QualityLivingArea, 후보 2는 세 feature 전체다.
- 각 후보는 split seeds 42, 2026, 3407의 5 folds, 총 15개 checkpoint를 쓴다.
- historical validation-change recipe의 Id=524, YearRemodAdd=2007 보정 정의를
  유지하지만 test에는 Id 524가 없어 실제 test 행 보정은 발생하지 않는다.
- 기존 학습 artifact만 로드하고 재학습하거나 후보끼리 예측 blend하지 않는다.
- Kaggle 점수는 생성이나 선택에 사용하지 않으며 public/private는 unverified다.


## Data

### 1. Load test, sample format, source metrics, and model definition


In [1]:
from __future__ import annotations

import csv
import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from torch import nn

ROOT = Path.cwd()
if not (ROOT / "data" / "test.csv").exists():
    ROOT = ROOT.parent.parent

TEST_PATH = ROOT / "data" / "test.csv"
SAMPLE_PATH = ROOT / "data" / "sample_submission.csv"
EXPERIMENTS_PATH = ROOT / "reports" / "experiments.csv"
NOTEBOOK_PATH = (
    ROOT / "notebooks" / "feature_engineering" /
    "basemlp_repeated_cv_feature_combo_submissions.ipynb"
)
SOURCE_SUMMARY_PATH = (
    ROOT / "reports" / "feature_engineering" /
    "basemlp_repeated_cv_feature_combo_summary.json"
)
REPORT_DIR = ROOT / "reports" / "feature_engineering"
MODEL_ROOT = (
    ROOT / "artifacts" / "feature_engineering" /
    "BASEMLP-20260720-FEAT-COMBOS-REPEATED-NB-12"
)
TEST_PREDICTIONS_PATH = (
    REPORT_DIR /
    "basemlp_repeated_cv_feature_combo_test_predictions.csv"
)
SUMMARY_PATH = (
    REPORT_DIR /
    "basemlp_repeated_cv_feature_combo_submission_summary.json"
)
SUBMISSION_DIR = ROOT / "submissions"

SOURCE_RUN_ID = "BASEMLP-20260720-FEAT-COMBOS-REPEATED-NB-12"
SOURCE_BASELINE_ID = "VALBASE-20260720-BASEMLP-REPEATED-3SEED-NB-03"
SPLIT_SEEDS = (42, 2026, 3407)
N_SPLITS = 5
HIDDEN_DIMS = (128, 64)
DEVICE = torch.device("cpu")

NUMERIC_COLUMNS = [
    "LotFrontage", "LotArea", "OverallQual", "OverallCond", "YearBuilt",
    "YearRemodAdd", "MasVnrArea", "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF",
    "TotalBsmtSF", "1stFlrSF", "2ndFlrSF", "LowQualFinSF", "GrLivArea",
    "BsmtFullBath", "BsmtHalfBath", "FullBath", "HalfBath", "BedroomAbvGr",
    "KitchenAbvGr", "TotRmsAbvGrd", "Fireplaces", "GarageYrBlt", "GarageCars",
    "GarageArea", "WoodDeckSF", "OpenPorchSF", "EnclosedPorch", "3SsnPorch",
    "ScreenPorch", "PoolArea", "MiscVal", "YrSold",
]
BASEMENT_COLUMNS = [
    "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2"
]
GARAGE_COLUMNS = ["GarageType", "GarageFinish", "GarageQual", "GarageCond"]

CANDIDATES = {
    "total_sf_quality_living_area": {
        "features": ("TotalSF", "QualityLivingArea"),
        "submission_id": (
            "BASEMLP-20260720-FEAT-COMBO-TSF-QLA-REPEATED-SUB-01"
        ),
    },
    "all_three": {
        "features": (
            "TotalSF", "TotalBathrooms", "QualityLivingArea"
        ),
        "submission_id": (
            "BASEMLP-20260720-FEAT-COMBO-ALL3-REPEATED-SUB-01"
        ),
    },
}


class BaseMLP(nn.Module):
    def __init__(self, input_dim: int) -> None:
        super().__init__()
        self.hidden1 = nn.Linear(input_dim, HIDDEN_DIMS[0])
        self.hidden2 = nn.Linear(HIDDEN_DIMS[0], HIDDEN_DIMS[1])
        self.output = nn.Linear(HIDDEN_DIMS[1], 1)
        self.activation = nn.ReLU()

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        hidden = self.activation(self.hidden1(inputs))
        hidden = self.activation(self.hidden2(hidden))
        return self.output(hidden).squeeze(1)


def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def predict_log_target(
    model: BaseMLP,
    features: np.ndarray,
    target_mean: float,
    target_std: float,
) -> np.ndarray:
    model.eval()
    with torch.no_grad():
        tensor = torch.as_tensor(
            features, dtype=torch.float32, device=DEVICE
        )
        standardized = model(tensor).cpu().numpy().astype(np.float64)
    return standardized * target_std + target_mean


def append_experiment(row: dict[str, str]) -> None:
    with EXPERIMENTS_PATH.open(newline="", encoding="utf-8") as handle:
        reader = csv.DictReader(handle)
        fieldnames = reader.fieldnames
        existing_ids = {
            existing["experiment_id"] for existing in reader
        }
    if fieldnames is None:
        raise RuntimeError("reports/experiments.csv has no header.")
    if row["experiment_id"] in existing_ids:
        return
    with EXPERIMENTS_PATH.open(
        "a", newline="", encoding="utf-8"
    ) as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writerow({
            field: row.get(field, "") for field in fieldnames
        })


REPORT_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)
test = pd.read_csv(TEST_PATH, keep_default_na=False)
sample = pd.read_csv(SAMPLE_PATH)
source_summary = json.loads(
    SOURCE_SUMMARY_PATH.read_text(encoding="utf-8")
)

assert test.shape == (1459, 80)
assert sample.shape == (1459, 2)
assert list(sample.columns) == ["Id", "SalePrice"]
assert sample["Id"].equals(test["Id"])
assert test["Id"].is_unique
assert sample["Id"].is_unique
assert sample["Id"].dtype == test["Id"].dtype
assert pd.api.types.is_numeric_dtype(sample["SalePrice"])

source_result_by_candidate = {
    row["candidate"]: row for row in source_summary["results"]
}
assert set(CANDIDATES).issubset(source_result_by_candidate)
assert source_summary["reference"]["id_524_yearremodadd"] == 2007
print(f"test: {test.shape[0]:,} rows × {test.shape[1]} columns")
print(f"sample: {sample.shape[0]:,} rows × {sample.shape[1]} columns")
print(f"test sha256: {sha256(TEST_PATH)}")
print("source candidate artifacts:", SOURCE_RUN_ID)


test: 1,459 rows × 80 columns
sample: 1,459 rows × 2 columns
test sha256: 8fdd3d829d4d986b58f845c9553b225e67dd8383624d90fb6ca1d4bed5798c1e
source candidate artifacts: BASEMLP-20260720-FEAT-COMBOS-REPEATED-NB-12


## Feature construction

### 2. Reproduce test preprocessing and the two feature sets


In [2]:
def clean_rows_historical_basemlp(
    frame: pd.DataFrame,
    categorical_columns: list[str],
) -> pd.DataFrame:
    cleaned = frame.copy()
    for column in NUMERIC_COLUMNS:
        cleaned[column] = pd.to_numeric(
            cleaned[column].replace({"NA": np.nan, "": np.nan}),
            errors="coerce",
        )

    cleaned.loc[
        cleaned["Id"].eq(524), "YearRemodAdd"
    ] = 2007

    basement_absent = cleaned["TotalBsmtSF"].fillna(0).eq(0)
    garage_absent = (
        cleaned["GarageCars"].fillna(0).eq(0)
        & cleaned["GarageArea"].fillna(0).eq(0)
    )
    for column in BASEMENT_COLUMNS:
        missing = cleaned[column].isin(["NA", ""])
        cleaned.loc[missing & basement_absent, column] = "NoBasement"
        cleaned.loc[missing & ~basement_absent, column] = "Unknown"
    for column in GARAGE_COLUMNS:
        missing = cleaned[column].isin(["NA", ""])
        cleaned.loc[missing & garage_absent, column] = "NoGarage"
        cleaned.loc[missing & ~garage_absent, column] = "Unknown"
    cleaned.loc[garage_absent, "GarageYrBlt"] = 0.0

    fireplace_absent = cleaned["Fireplaces"].fillna(0).eq(0)
    pool_absent = cleaned["PoolArea"].fillna(0).eq(0)
    cleaned.loc[
        cleaned["FireplaceQu"].isin(["NA", ""]) & fireplace_absent,
        "FireplaceQu",
    ] = "NoFireplace"
    cleaned.loc[
        cleaned["PoolQC"].isin(["NA", ""]) & pool_absent,
        "PoolQC",
    ] = "NoPool"

    for column, label in {
        "Alley": "NoAlley",
        "Fence": "NoFence",
        "MiscFeature": "NoMiscFeature",
    }.items():
        cleaned[column] = cleaned[column].replace(
            {"NA": label, "": label}
        )

    cleaned["MSSubClass"] = cleaned["MSSubClass"].map(
        lambda value: f"SC{value}"
    )
    cleaned["MoSold"] = cleaned["MoSold"].map(
        lambda value: f"M{int(value):02d}"
    )
    for column in categorical_columns:
        cleaned[column] = cleaned[column].replace(
            {"NA": "Unknown", "": "Unknown"}
        )
    return cleaned


def add_engineered_features(
    frame: pd.DataFrame,
    feature_names: tuple[str, ...],
) -> pd.DataFrame:
    featured = frame.copy()
    if "TotalSF" in feature_names:
        featured["TotalSF"] = (
            featured[
                ["TotalBsmtSF", "1stFlrSF", "2ndFlrSF"]
            ].fillna(0).sum(axis=1)
        )
    if "TotalBathrooms" in feature_names:
        featured["TotalBathrooms"] = (
            featured["FullBath"].fillna(0)
            + 0.5 * featured["HalfBath"].fillna(0)
            + featured["BsmtFullBath"].fillna(0)
            + 0.5 * featured["BsmtHalfBath"].fillna(0)
        )
    if "QualityLivingArea" in feature_names:
        featured["QualityLivingArea"] = (
            featured["OverallQual"] * featured["GrLivArea"]
        )
    return featured


categorical_columns = [
    column
    for column in test.columns
    if column not in {*NUMERIC_COLUMNS, "Id"}
]
assert len(categorical_columns) == 45
clean_test = clean_rows_historical_basemlp(
    test, categorical_columns
)
candidate_test_frames = {}
audit_rows = []
for candidate, config in CANDIDATES.items():
    featured_test = add_engineered_features(
        clean_test, config["features"]
    )
    assert featured_test.drop(
        columns=list(config["features"])
    ).equals(clean_test)
    for feature_name in config["features"]:
        assert featured_test[feature_name].notna().all()
        assert np.isfinite(featured_test[feature_name]).all()
        assert featured_test[feature_name].ge(0).all()
    candidate_test_frames[candidate] = featured_test
    audit_rows.append({
        "candidate": candidate,
        "features": " + ".join(config["features"]),
        "test_rows": len(featured_test),
        "original_columns_modified": 0,
        "generated_feature_count": len(config["features"]),
    })
display(pd.DataFrame(audit_rows))


,candidate,features,test_rows,original_columns_modified,generated_feature_count
0,total_sf_quality_living_area,TotalSF + QualityLivingArea,1459,0,2
1,all_three,TotalSF + TotalBathrooms + QualityLivingArea,1459,0,3


## Inference

### 3. Restore 15 checkpoints per candidate and predict in log space


In [3]:
candidate_log_prediction_matrices = {}
prediction_report = pd.DataFrame({
    "Id": test["Id"].to_numpy(dtype=np.int64)
})
checkpoint_rows = []

for candidate, config in CANDIDATES.items():
    candidate_dir = MODEL_ROOT / candidate
    checkpoint_dir = candidate_dir / "checkpoints"
    preprocessor_dir = candidate_dir / "preprocessors"
    checkpoint_files = sorted(checkpoint_dir.glob("*.pt"))
    preprocessor_files = sorted(
        preprocessor_dir.glob("*.joblib")
    )
    assert len(checkpoint_files) == 15
    assert len(preprocessor_files) == 15

    fold_log_predictions = []
    expected_source_experiment = (
        f"{SOURCE_RUN_ID}-{candidate.upper()}"
    )
    for split_seed in SPLIT_SEEDS:
        for fold in range(1, N_SPLITS + 1):
            checkpoint_path = (
                checkpoint_dir /
                f"seed_{split_seed}_fold_{fold}_best.pt"
            )
            preprocessor_path = (
                preprocessor_dir /
                f"seed_{split_seed}_fold_{fold}_preprocessor.joblib"
            )
            assert checkpoint_path.exists()
            assert preprocessor_path.exists()

            checkpoint = torch.load(
                checkpoint_path,
                map_location=DEVICE,
                weights_only=False,
            )
            preprocessor = joblib.load(preprocessor_path)
            transformed_test = preprocessor.transform(
                candidate_test_frames[candidate]
            ).astype(np.float32)
            assert transformed_test.shape[1] == checkpoint["input_dim"]
            assert checkpoint["experiment_id"] == expected_source_experiment
            assert checkpoint["seed"] == split_seed + fold
            assert checkpoint["target_std"] > 0

            model = BaseMLP(checkpoint["input_dim"]).to(DEVICE)
            model.load_state_dict(checkpoint["model_state_dict"])
            fold_log_prediction = predict_log_target(
                model,
                transformed_test,
                checkpoint["target_mean"],
                checkpoint["target_std"],
            )
            assert np.isfinite(fold_log_prediction).all()
            fold_log_predictions.append(fold_log_prediction)
            prediction_report[
                f"{candidate}__seed_{split_seed}_fold_{fold}_log"
            ] = fold_log_prediction
            checkpoint_rows.append({
                "candidate": candidate,
                "split_seed": split_seed,
                "fold": fold,
                "checkpoint": str(
                    checkpoint_path.relative_to(ROOT)
                ),
                "preprocessor": str(
                    preprocessor_path.relative_to(ROOT)
                ),
                "input_dim": checkpoint["input_dim"],
            })

    prediction_matrix = np.column_stack(fold_log_predictions)
    assert prediction_matrix.shape == (len(test), 15)
    candidate_log_prediction_matrices[candidate] = (
        prediction_matrix
    )
    prediction_report[
        f"{candidate}__mean_log_prediction"
    ] = prediction_matrix.mean(axis=1)

checkpoint_audit = pd.DataFrame(checkpoint_rows)
assert len(checkpoint_audit) == 30
display(
    checkpoint_audit.groupby("candidate").agg(
        checkpoints=("checkpoint", "nunique"),
        preprocessors=("preprocessor", "nunique"),
        input_dim_min=("input_dim", "min"),
        input_dim_max=("input_dim", "max"),
    )
)


,checkpoints,preprocessors,input_dim_min,input_dim_max
candidate,,,,
all_three,15,15,326,331
total_sf_quality_living_area,15,15,325,330


## Results

### 4. Write and validate two submission CSV files


In [4]:
submission_rows = []
submission_frames = {}

for candidate, config in CANDIDATES.items():
    mean_log_prediction = (
        candidate_log_prediction_matrices[candidate].mean(axis=1)
    )
    sale_price_prediction = np.expm1(mean_log_prediction)
    submission = sample.copy()
    submission["Id"] = test["Id"].to_numpy()
    submission["SalePrice"] = sale_price_prediction.astype(
        sample["SalePrice"].dtype
    )
    submission_path = (
        SUBMISSION_DIR /
        f"submission_{config['submission_id']}.csv"
    )
    submission.to_csv(submission_path, index=False)

    reloaded = pd.read_csv(submission_path)
    assert list(reloaded.columns) == list(sample.columns)
    assert reloaded.shape == sample.shape
    assert reloaded["Id"].equals(test["Id"])
    assert reloaded["Id"].is_unique
    assert not reloaded.isna().any().any()
    assert pd.api.types.is_numeric_dtype(reloaded["SalePrice"])
    assert np.isfinite(reloaded["SalePrice"]).all()
    assert reloaded["SalePrice"].gt(0).all()
    assert reloaded.dtypes.to_dict() == sample.dtypes.to_dict()

    submission_frames[candidate] = reloaded
    source_result = source_result_by_candidate[candidate]
    submission_rows.append({
        "candidate": candidate,
        "features": " + ".join(config["features"]),
        "submission_id": config["submission_id"],
        "submission_path": str(
            submission_path.relative_to(ROOT)
        ),
        "submission_sha256": sha256(submission_path),
        "rows": len(reloaded),
        "prediction_min": float(reloaded["SalePrice"].min()),
        "prediction_median": float(
            reloaded["SalePrice"].median()
        ),
        "prediction_max": float(reloaded["SalePrice"].max()),
        "cv_mean": source_result["cv_mean"],
        "cv_std": source_result["cv_std"],
        "three_seed_ensemble_oof_rmsle": source_result[
            "three_seed_ensemble_oof_rmsle"
        ],
    })

prediction_report.to_csv(TEST_PREDICTIONS_PATH, index=False)
submission_results = pd.DataFrame(submission_rows)

first = submission_frames["total_sf_quality_living_area"][
    "SalePrice"
].to_numpy(dtype=np.float64)
second = submission_frames["all_three"][
    "SalePrice"
].to_numpy(dtype=np.float64)
absolute_difference = np.abs(first - second)
max_difference_index = int(np.argmax(absolute_difference))
pairwise_comparison = {
    "different_rows": int(np.count_nonzero(first != second)),
    "mean_absolute_difference": float(
        absolute_difference.mean()
    ),
    "median_absolute_difference": float(
        np.median(absolute_difference)
    ),
    "max_absolute_difference": float(
        absolute_difference.max()
    ),
    "max_difference_id": int(
        test["Id"].iloc[max_difference_index]
    ),
    "total_sf_quality_living_area_at_max": float(
        first[max_difference_index]
    ),
    "all_three_at_max": float(second[max_difference_index]),
    "max_difference_test_context": {
        column: float(test[column].iloc[max_difference_index])
        for column in [
            "GrLivArea", "OverallQual", "TotalBsmtSF",
            "1stFlrSF", "2ndFlrSF"
        ]
    },
    "pearson_correlation": float(np.corrcoef(first, second)[0, 1]),
}
assert pairwise_comparison["different_rows"] == len(test)

run_timestamp = datetime.now(timezone.utc).isoformat()
summary = {
    "run_timestamp_utc": run_timestamp,
    "role": "submission generation for the two leading repeated-CV feature sets",
    "source_run_id": SOURCE_RUN_ID,
    "source_baseline_id": SOURCE_BASELINE_ID,
    "method": {
        "models_per_submission": 15,
        "split_seeds": list(SPLIT_SEEDS),
        "folds_per_seed": N_SPLITS,
        "aggregation": (
            "mean of 15 restored-checkpoint log predictions, "
            "then expm1"
        ),
        "prediction_blend_between_candidates": False,
        "id_524_yearremodadd_recipe": 2007,
    },
    "input_checks": {
        "test_path": "data/test.csv",
        "test_sha256": sha256(TEST_PATH),
        "sample_path": "data/sample_submission.csv",
        "sample_sha256": sha256(SAMPLE_PATH),
        "test_rows": len(test),
        "sample_columns": list(sample.columns),
        "id_order_matches": bool(sample["Id"].equals(test["Id"])),
        "sample_dtypes": {
            column: str(dtype)
            for column, dtype in sample.dtypes.items()
        },
    },
    "submissions": json.loads(
        submission_results.to_json(orient="records")
    ),
    "pairwise_comparison": pairwise_comparison,
    "validation": {
        "checkpoint_count": len(checkpoint_audit),
        "unique_checkpoint_count": int(
            checkpoint_audit["checkpoint"].nunique()
        ),
        "unique_preprocessor_count": int(
            checkpoint_audit["preprocessor"].nunique()
        ),
        "submission_format_checks_passed": True,
        "missing_predictions": 0,
        "nonfinite_predictions": 0,
        "nonpositive_predictions": 0,
        "external_project_script_imports": 0,
    },
    "artifacts": {
        "notebook": str(NOTEBOOK_PATH.relative_to(ROOT)),
        "test_fold_predictions": str(
            TEST_PREDICTIONS_PATH.relative_to(ROOT)
        ),
        "summary": str(SUMMARY_PATH.relative_to(ROOT)),
    },
    "kaggle": {
        "public_scores": "unverified",
        "private_scores": "unverified",
    },
}
SUMMARY_PATH.write_text(
    json.dumps(summary, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

for candidate, config in CANDIDATES.items():
    source_result = source_result_by_candidate[candidate]
    submission_path = (
        SUBMISSION_DIR /
        f"submission_{config['submission_id']}.csv"
    )
    append_experiment({
        "experiment_id": config["submission_id"],
        "datetime": run_timestamp,
        "objective": (
            "Create a repeated-CV BaseMLP submission for "
            + " + ".join(config["features"])
        ),
        "data_version": (
            f"test sha256={sha256(TEST_PATH)}; "
            f"sample sha256={sha256(SAMPLE_PATH)}"
        ),
        "split_strategy": (
            "KFold(5,shuffle=True) repeated at seeds 42|2026|3407"
        ),
        "folds": "5x3",
        "seed": "42|2026|3407; model seed=split_seed+fold",
        "metric": "RMSLE / RMSE on log1p target",
        "preprocessing": (
            "Load 15 fold-fitted historical BaseMLP preprocessors; "
            "Id=524 YearRemodAdd=2007 recipe retained"
        ),
        "features": (
            "79 original predictors + "
            + " + ".join(config["features"])
            + "; Id excluded"
        ),
        "model": (
            "Manual PyTorch BaseMLP; 15 restored-checkpoint "
            "mean-log inference; no cross-candidate blend"
        ),
        "architecture": (
            "input->128(ReLU)->64(ReLU)->1"
        ),
        "optimizer": "Adam(weight_decay=0) in source training",
        "loss": (
            "MSELoss on fold-standardized log1p(SalePrice) "
            "in source training"
        ),
        "learning_rate": 0.001,
        "batch_size": 64,
        "max_epochs": 200,
        "patience": 25,
        "hyperparameters": json.dumps({
            "source_run_id": SOURCE_RUN_ID,
            "source_candidate": candidate,
            "generated_features": list(config["features"]),
            "models_per_submission": 15,
            "test_prediction_strategy": (
                "mean of 15 log predictions then expm1"
            ),
            "prediction_blend_between_candidates": False,
        }),
        "cv_mean": source_result["cv_mean"],
        "cv_std": source_result["cv_std"],
        "holdout_score": source_result[
            "three_seed_ensemble_oof_rmsle"
        ],
        "checkpoint_path": str(
            (MODEL_ROOT / candidate / "checkpoints").relative_to(ROOT)
        ),
        "artifact_path": " | ".join([
            str(NOTEBOOK_PATH.relative_to(ROOT)),
            str(submission_path.relative_to(ROOT)),
            str(TEST_PREDICTIONS_PATH.relative_to(ROOT)),
            str(SUMMARY_PATH.relative_to(ROOT)),
        ]),
        "result": "submission_candidate_public_unverified",
        "interpretation": (
            "Submission format and 15-checkpoint traceability "
            "verified; Kaggle public/private scores unverified."
        ),
        "next_action": (
            "Submit this exact CSV if chosen and record the "
            "observed Kaggle score without tuning against it."
        ),
    })

display(submission_results)
display(pd.Series(pairwise_comparison, name="pairwise prediction comparison"))
print(f"Summary: {SUMMARY_PATH.relative_to(ROOT)}")


,candidate,features,submission_id,submission_path,submission_sha256,rows,prediction_min,prediction_median,prediction_max,cv_mean,cv_std,three_seed_ensemble_oof_rmsle
0,total_sf_quality_living_area,TotalSF + QualityLivingArea,BASEMLP-20260720-FEAT-COMBO-TSF-QLA-REPEATED-S...,submissions/submission_BASEMLP-20260720-FEAT-C...,385b03d223656aaad6438e58bb7bb66c021401e2c50278...,1459,47607.905463,156588.403860,864169.805683,0.143928,0.031134,0.131811
1,all_three,TotalSF + TotalBathrooms + QualityLivingArea,BASEMLP-20260720-FEAT-COMBO-ALL3-REPEATED-SUB-01,submissions/submission_BASEMLP-20260720-FEAT-C...,b4e73d4f106686c11eff1719a7eba8a829e281fb1765ef...,1459,48630.983312,156320.432692,617470.443624,0.144413,0.024417,0.132963


different_rows                                                                      1459
mean_absolute_difference                                                     3323.010885
median_absolute_difference                                                   2246.142592
max_absolute_difference                                                    246699.362058
max_difference_id                                                                   2550
total_sf_quality_living_area_at_max                                        864169.805683
all_three_at_max                                                           617470.443624
max_difference_test_context            {'GrLivArea': 5095.0, 'OverallQual': 10.0, 'To...
pearson_correlation                                                             0.994616
Name: pairwise prediction comparison, dtype: object

Summary: reports/feature_engineering/basemlp_repeated_cv_feature_combo_submission_summary.json


## Takeaways

- TotalSF + QualityLivingArea CSV의 예측 범위는 $47,608~$864,170이고,
  세 feature 전체 CSV는 $48,631~$617,470이다.
- 두 파일은 1,459행 모두 다르지만 중앙 절대 차이는 $2,246, 상관은 0.9946이다.
- 최대 차이는 대면적 tail인 Id 2550에서 발생했다: $864,170 대 $617,470.
- 열, 행 수, Id 순서, dtype과 예측 유한값·양수 조건을 모두 통과했다.
- Kaggle public/private 점수는 제출 전까지 unverified다.
